In [1]:
# BLOCK A: 全域設定 (CONFIG) - 所有模組開關都在這裡控制
# ============================================================
import os
import re
import ast
import random
import copy
import glob
import json
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.transforms import InterpolationMode
from PIL import Image, ImageDraw
import timm
import time
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, classification_report, confusion_matrix

CONFIG = {
    # ========================================================
    # 0. 實驗識別
    # ========================================================
    # ★ 新實驗名稱一定要換，避免讀到 ConvNeXt-Tiny 舊 checkpoint
    "EXPERIMENT_NAME": "cbam_resnet18_age_sex_liu2026",

    # ========================================================
    # 1. 路徑設定
    # ========================================================
    "META_CSV": "/kaggle/input/datasets/khyeh0719/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/ptbxl_database.csv",
    "SCP_CSV": "/kaggle/input/datasets/khyeh0719/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/scp_statements.csv",
    "IMG_ROOT": "/kaggle/input/datasets/bjoernjostein/ptb-xl-ecg-image-gmc2024",
    "IMG_RATE": "lr",
    "MAX_IMG_PER_RECORD": 1,

    "OUTPUT_DIR": "/kaggle/working/outputs",
    "CHECKPOINT_DIR": "/kaggle/working/checkpoints",

    # ========================================================
    # 2. Kaggle 12 小時續跑 / Checkpoint
    # ========================================================
    "RESUME_TRAINING": True,
    "RESUME_CHECKPOINT": None,

    "RESUME_SEARCH_ROOTS": [
        "/kaggle/working",
        "/kaggle/input",
    ],

    "SAVE_CHECKPOINT_EVERY": 1,
    "SAVE_HISTORY_EVERY_EPOCH": True,
    "STRICT_RESUME_CONFIG": True,

    "GRACEFUL_STOP_BEFORE_KAGGLE_LIMIT": True,
    "KAGGLE_SESSION_SOFT_LIMIT_HOURS": 10.5,

    # ========================================================
    # 3. 任務設定
    # ========================================================
    "SINGLE_LABEL_ONLY": True,
    "AGE_CLIP_MAX": 90,

    # ========================================================
    # 4. 多模態設定
    # ========================================================
    # 與上一輪 ConvNeXt-Tiny 相同：ECG Image + Age + Sex
    "USE_AGE_MODALITY": True,
    "USE_SEX_MODALITY": True,
    "DEMO_EMBED_DIM": 16,

    # ========================================================
    # 5. 模型設定
    # ========================================================
    # ★ 本輪唯一主要架構變因：ConvNeXt-Tiny → CBAM-ResNet18
    "BACKBONE": "cbam_resnet18",

    # CBAM-ResNet18 不使用 ConvNeXt 的 stochastic depth。
    # 保留欄位是為了共用程式介面，但設為 0。
    "DROP_PATH_RATE": 0.0,

    # Standard CBAM 超參數
    "CBAM_REDUCTION": 16,
    "CBAM_SPATIAL_KERNEL": 7,

    "PRETRAINED": True,
    "NUM_CLASSES": 5,
    "IMG_SIZE": 224,
    "MASK_TOP_RATIO": 0.30,

    # Grad-CAM 對 CBAM-ResNet18 改抓最後一個 residual block 的 conv2
    "GRADCAM_TARGET_LAYER": "layer4.1.conv2",

    # ========================================================
    # 6. ECG IMAGE DATA AUGMENTATION
    # ========================================================
    "USE_AUGMENTATION": True,

    # 與上一輪相同：Liu-style conservative augmentation
    "AUG_PROFILE": "liu2026",
    "AUG_ROTATION_DEGREE": 3.0,
    "AUG_TRANSLATE": (0.05, 0.05),

    # 以下 profile 在 liu2026 不會啟用，保留給其他 ablation
    "AUG_SCALE": (0.95, 1.05),
    "AUG_BRIGHTNESS": 0.10,
    "AUG_CONTRAST": 0.10,
    "AUG_PERSPECTIVE_DISTORTION": 0.08,
    "AUG_PERSPECTIVE_P": 0.10,
    "AUG_BLUR_KERNEL_SIZE": 3,
    "AUG_BLUR_SIGMA": (0.1, 1.0),
    "AUG_BLUR_P": 0.10,
    "AUG_SHEAR": 3.0,
    "AUG_RANDOM_ERASING_P": 0.20,
    "AUG_HORIZONTAL_FLIP_P": 0.0,
    "AUG_VERTICAL_FLIP_P": 0.0,

    "NORMALIZE_MEAN": [0.485, 0.456, 0.406],
    "NORMALIZE_STD": [0.229, 0.224, 0.225],

    # ========================================================
    # 7. 訓練設定
    # ========================================================
    # 為公平比較，以下沿用 ConvNeXt-Tiny 設定
    "BATCH_SIZE": 32,
    "NUM_WORKERS": 2,
    "EPOCHS": 50,

    "FREEZE_EPOCHS": 5,
    "LR_HEAD": 5e-4,
    "LR_BACKBONE": 3e-5,
    "WEIGHT_DECAY": 1e-3,
    "SEED": 42,

    # ========================================================
    # 8. Scheduler
    # ========================================================
    "SCHEDULER_TYPE": "plateau",

    # ========================================================
    # 9. Class imbalance
    # ========================================================
    "USE_WEIGHTED_SAMPLER": False,
    "USE_CLASS_WEIGHTED_LOSS": True,
    "USE_FOCAL_LOSS": False,
    "FOCAL_GAMMA": 2.0,

    # ========================================================
    # 10. Mixup
    # ========================================================
    "USE_MIXUP": False,
    "MIXUP_ALPHA": 0.2,

    # ========================================================
    # 11. Grad-CAM
    # ========================================================
    "USE_GRADCAM": True,

    # ========================================================
    # 12. Early stopping
    # ========================================================
    "EARLY_STOP_METRIC": "val_f1_macro",
    "EARLY_STOP_PATIENCE": 10,

    # ========================================================
    # 13. Log
    # ========================================================
    "VERBOSE_BATCH": True,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
os.makedirs(CONFIG["CHECKPOINT_DIR"], exist_ok=True)

print("=" * 70)
print("CONFIG")
print("=" * 70)
print("Device                :", DEVICE)
print("Experiment            :", CONFIG["EXPERIMENT_NAME"])
print("Backbone              :", CONFIG["BACKBONE"])
print("CBAM reduction        :", CONFIG["CBAM_REDUCTION"])
print("CBAM spatial kernel   :", CONFIG["CBAM_SPATIAL_KERNEL"])
print("Image size            :", CONFIG["IMG_SIZE"])
print("Age modality          :", CONFIG["USE_AGE_MODALITY"])
print("Sex modality          :", CONFIG["USE_SEX_MODALITY"])
print("Augmentation profile  :", CONFIG["AUG_PROFILE"])
print("Rotation              : ±{}°".format(CONFIG["AUG_ROTATION_DEGREE"]))
print("Translation           : ±{}%".format(int(CONFIG["AUG_TRANSLATE"][0] * 100)))
print("Weighted sampler      :", CONFIG["USE_WEIGHTED_SAMPLER"])
print("Class-weighted loss   :", CONFIG["USE_CLASS_WEIGHTED_LOSS"])
print("Batch size            :", CONFIG["BATCH_SIZE"])
print("Freeze epochs         :", CONFIG["FREEZE_EPOCHS"])
print("LR head               :", CONFIG["LR_HEAD"])
print("LR backbone           :", CONFIG["LR_BACKBONE"])
print("Scheduler             :", CONFIG["SCHEDULER_TYPE"])
print("Early stop metric     :", CONFIG["EARLY_STOP_METRIC"])
print("Early stop patience   :", CONFIG["EARLY_STOP_PATIENCE"])
print("Target epochs         :", CONFIG["EPOCHS"])
print("Resume training       :", CONFIG["RESUME_TRAINING"])
print("Grad-CAM target       :", CONFIG["GRADCAM_TARGET_LAYER"])
print("=" * 70)
print("A")



CONFIG
Device                : cuda
Experiment            : cbam_resnet18_age_sex_liu2026
Backbone              : cbam_resnet18
CBAM reduction        : 16
CBAM spatial kernel   : 7
Image size            : 224
Age modality          : True
Sex modality          : True
Augmentation profile  : liu2026
Rotation              : ±3.0°
Translation           : ±5%
Weighted sampler      : False
Class-weighted loss   : True
Batch size            : 32
Freeze epochs         : 5
LR head               : 0.0005
LR backbone           : 3e-05
Scheduler             : plateau
Early stop metric     : val_f1_macro
Early stop patience   : 10
Target epochs         : 50
Resume training       : True
Grad-CAM target       : layer4.1.conv2
A


In [2]:
# BLOCK B: 隨機種子固定 (可重現性)
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["SEED"])

print('B')

B


In [3]:
# BLOCK C: PTB-XL Metadata 讀取與 diagnostic superclass 聚合
# ============================================================
def load_ptbxl_metadata(meta_csv: str, scp_csv: str, single_label_only: bool = True) -> pd.DataFrame:
    meta = pd.read_csv(meta_csv, index_col="ecg_id")
    meta["scp_codes"] = meta["scp_codes"].apply(ast.literal_eval)

    agg_df = pd.read_csv(scp_csv, index_col=0)
    agg_df = agg_df[agg_df.diagnostic == 1]

    def aggregate_diagnostic(scp_codes: dict):
        classes = set()
        for code in scp_codes.keys():
            if code in agg_df.index:
                classes.add(agg_df.loc[code].diagnostic_class)
        return list(classes)

    meta["diagnostic_superclass"] = meta["scp_codes"].apply(aggregate_diagnostic)
    meta = meta[meta["diagnostic_superclass"].apply(len) > 0].copy()

    if single_label_only:
        meta = meta[meta["diagnostic_superclass"].apply(len) == 1].copy()
        meta["label"] = meta["diagnostic_superclass"].apply(lambda x: x[0])
    else:
        # 多標籤模式：保留 list，訓練時需搭配 BCEWithLogitsLoss (本檔案預設走單標籤路徑)
        meta["label"] = meta["diagnostic_superclass"]

    return meta

print('C')

C


In [4]:
# BLOCK D: 年齡清理 + 影像路徑展開 (含資料夾分層規則)
# ============================================================
def get_ptbxl_folder(ecg_id: int) -> str:
    """ecg_id -> 千位分組資料夾名稱，例如 12345 -> '12000'"""
    return f"{(ecg_id // 1000) * 1000:05d}"


def clean_age(meta: pd.DataFrame, clip_max: int) -> pd.DataFrame:
    meta = meta.copy()

    n_before = len(meta)
    meta = meta[meta["age"].notna()].copy()
    n_dropped = n_before - len(meta)
    if n_dropped > 0:
        print(f"[clean_age] 移除 {n_dropped} 筆缺少年齡的紀錄 ({n_before} -> {len(meta)})")

    # PTB-XL 對 >89 歲的紀錄設為 300 歲 (HIPAA 去識別化)，需特別處理
    meta["is_elderly_capped"] = meta["age"] >= 200
    meta["age_clean"] = meta["age"].clip(upper=clip_max)
    return meta

def clean_sex(meta: pd.DataFrame) -> pd.DataFrame:
    """PTB-XL sex 欄位: 0=男性, 1=女性"""
    meta = meta.copy()
    n_before = len(meta)
    meta = meta[meta["sex"].notna()].copy()
    n_dropped = n_before - len(meta)
    if n_dropped > 0:
        print(f"[clean_sex] 移除 {n_dropped} 筆缺少性別的紀錄 ({n_before} -> {len(meta)})")
    meta["sex_clean"] = meta["sex"].astype(int)
    return meta


def build_image_index(meta_df: pd.DataFrame, img_root: str, rate: str,
                       max_img_per_record: int) -> pd.DataFrame:
    rows = []
    for ecg_id, row in meta_df.iterrows():
        folder = get_ptbxl_folder(ecg_id)
        for idx in range(max_img_per_record):
            fname = f"{ecg_id:05d}_{rate}-{idx}.png"
            fpath = os.path.join(img_root, folder, fname)
            if os.path.exists(fpath):
                rows.append({
                    "img_path": fpath,
                    "ecg_id": ecg_id,
                    "patient_id": row["patient_id"],
                    "age_clean": row["age_clean"],
                    "sex_clean": row["sex_clean"],
                    "label": row["label"],
                })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(
            "沒有找到任何符合命名規則的影像檔案，請先確認 IMG_ROOT / IMG_RATE / 檔名格式是否正確。"
        )
    return df

print('D')

D


In [5]:
# BLOCK E: Patient-level train/val/test 切分 (避免資料洩漏)
# ============================================================
def patient_level_split(df: pd.DataFrame, seed: int, val_size=0.15, test_size=0.15):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(gss1.split(df, groups=df["patient_id"]))
    trainval_df = df.iloc[trainval_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    relative_val_size = val_size / (1 - test_size)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=relative_val_size, random_state=seed)
    train_idx, val_idx = next(gss2.split(trainval_df, groups=trainval_df["patient_id"]))
    train_df = trainval_df.iloc[train_idx].reset_index(drop=True)
    val_df = trainval_df.iloc[val_idx].reset_index(drop=True)

    # 檢查 patient_id 不重疊
    assert set(train_df.patient_id) & set(val_df.patient_id) == set()
    assert set(train_df.patient_id) & set(test_df.patient_id) == set()
    assert set(val_df.patient_id) & set(test_df.patient_id) == set()

    return train_df, val_df, test_df


print('E')

E


In [6]:
# BLOCK F: 年齡標準化 (用 train set 統計量，避免資訊洩漏)
# ============================================================
class AgeScaler:
    def __init__(self):
        self.mean = None
        self.std = None

    def fit(self, ages: np.ndarray):
        self.mean = float(np.mean(ages))
        self.std = float(np.std(ages) + 1e-6)
        return self

    def transform(self, ages: np.ndarray):
        return (ages - self.mean) / self.std

print('F')

F


In [7]:
# BLOCK G: ECG 影像資料增強 (Template-based Augmentation)
# ============================================================


class MaskTopRegion:
    """
    遮蔽 ECG 影像最上方一定比例。

    目的：
    避免模型學到病人編號、日期、文字、機器資訊等
    非 ECG waveform 的 shortcut features。
    """

    def __init__(
        self,
        top_ratio: float = 0.2,
        fill_color=(255, 255, 255)
    ):
        self.top_ratio = top_ratio
        self.fill_color = fill_color


    def __call__(self, img):

        img = img.copy()

        w, h = img.size

        mask_height = int(
            h * self.top_ratio
        )

        draw = ImageDraw.Draw(img)

        draw.rectangle(
            [
                0,
                0,
                w,
                mask_height
            ],
            fill=self.fill_color
        )

        return img



# ============================================================
# 顯示目前 augmentation 設定
# ============================================================

def _print_augmentation_profile(cfg: dict):

    """
    訓練開始前印出目前 augmentation profile。

    方便 Kaggle 每次實驗留下紀錄。
    """

    profile = str(
        cfg.get(
            "AUG_PROFILE",
            "none"
        )
    ).lower()


    enabled = (
        bool(
            cfg.get(
                "USE_AUGMENTATION",
                True
            )
        )
        and
        profile != "none"
    )


    print(
        "\n"
        +
        "=" * 66
    )

    print(
        "ECG IMAGE AUGMENTATION CONFIG"
    )

    print(
        "=" * 66
    )

    print(
        f"Enabled            : {enabled}"
    )

    print(
        f"Profile            : {profile}"
    )


    # ========================================================
    # Augmentation OFF
    # ========================================================

    if not enabled:

        print(
            "Train augmentation : OFF "
            "(train = eval preprocessing)"
        )


    # ========================================================
    # ORIGINAL
    # ========================================================

    elif profile == "original":

        print(
            "Rotation           : ±5°"
        )

        print(
            "Translation        : ±5%"
        )

        print(
            "Scale              : 0.90–1.10"
        )

        print(
            "Shear              : ±3°"
        )

        print(
            "Brightness/Contrast: ±0.15"
        )

        print(
            "Random Erasing     : p=0.20"
        )


    # ========================================================
    # LIU 2026
    # ========================================================

    elif profile == "liu2026":

        print(
            f"Rotation           : "
            f"±{cfg['AUG_ROTATION_DEGREE']}°"
        )

        print(
            f"Translation        : "
            f"±{cfg['AUG_TRANSLATE'][0] * 100:.1f}% / "
            f"±{cfg['AUG_TRANSLATE'][1] * 100:.1f}%"
        )


    # ========================================================
    # SAFE / REAL WORLD
    # ========================================================

    elif profile in (
        "safe",
        "real_world"
    ):

        print(
            f"Rotation           : "
            f"±{cfg['AUG_ROTATION_DEGREE']}°"
        )

        print(
            f"Translation        : "
            f"±{cfg['AUG_TRANSLATE'][0] * 100:.1f}% / "
            f"±{cfg['AUG_TRANSLATE'][1] * 100:.1f}%"
        )

        print(
            f"Scale              : "
            f"{cfg['AUG_SCALE'][0]:.2f}–"
            f"{cfg['AUG_SCALE'][1]:.2f}"
        )

        print(
            f"Brightness         : "
            f"±{cfg['AUG_BRIGHTNESS']:.2f}"
        )

        print(
            f"Contrast           : "
            f"±{cfg['AUG_CONTRAST']:.2f}"
        )


        # ----------------------------------------------------
        # REAL WORLD additional augmentation
        # ----------------------------------------------------

        if profile == "real_world":

            print(
                f"Perspective        : "
                f"distortion="
                f"{cfg['AUG_PERSPECTIVE_DISTORTION']}, "
                f"p="
                f"{cfg['AUG_PERSPECTIVE_P']}"
            )

            print(
                f"Gaussian Blur      : "
                f"p="
                f"{cfg['AUG_BLUR_P']}"
            )


    # ========================================================
    # ECG Flip warning
    # ========================================================

    print(
        f"Horizontal Flip    : "
        f"p="
        f"{cfg.get('AUG_HORIZONTAL_FLIP_P', 0.0)} "
        f"(ECG 不建議)"
    )

    print(
        f"Vertical Flip      : "
        f"p="
        f"{cfg.get('AUG_VERTICAL_FLIP_P', 0.0)} "
        f"(ECG 不建議)"
    )

    print(
        "=" * 66
        +
        "\n"
    )



# ============================================================
# 建立 Train / Validation Transform
# ============================================================

def get_transforms(cfg: dict):

    """
    依照 BLOCK A 的 AUG_PROFILE
    建立 train / validation / test transform。


    Profile
    ----------------------------------------------------------

    none
        不做 augmentation。

    original
        完整重現 20260725 原版 augmentation。

    liu2026
        Rotation ±3°
        Translation ±5%

    safe
        ★目前推薦 baseline

        Rotation ±3°
        Translation ±5%
        Scale 0.95–1.05
        Brightness ±0.10
        Contrast ±0.10

    real_world
        SAFE +
        Perspective +
        slight Gaussian Blur
    """


    # ========================================================
    # 基本設定
    # ========================================================

    img_size = cfg[
        "IMG_SIZE"
    ]


    profile = str(
        cfg.get(
            "AUG_PROFILE",
            "none"
        )
    ).lower()


    use_aug = (

        bool(
            cfg.get(
                "USE_AUGMENTATION",
                True
            )
        )

        and

        profile != "none"

    )


    # ========================================================
    # 檢查 profile 名稱
    # ========================================================

    valid_profiles = {

        "none",

        "original",

        "liu2026",

        "safe",

        "real_world"

    }


    if profile not in valid_profiles:

        raise ValueError(

            f"Unknown AUG_PROFILE="
            f"'{profile}'. "

            f"請使用 "
            f"{sorted(valid_profiles)}"

        )


    # ========================================================
    # Top mask
    # ========================================================

    mask_top = MaskTopRegion(

        top_ratio=cfg[
            "MASK_TOP_RATIO"
        ],

        fill_color=(
            255,
            255,
            255
        )

    )


    # ========================================================
    # Train / Val 共用 preprocessing
    # ========================================================

    base_pre = [

        mask_top,

        transforms.Resize(

            (
                img_size,
                img_size
            ),

            interpolation=
            InterpolationMode.BILINEAR

        ),

    ]


    # ========================================================
    # Train transform list
    # ========================================================

    train_tf_list = list(
        base_pre
    )


    # ========================================================
    # Augmentation ON
    # ========================================================

    if use_aug:


        # ====================================================
        # PROFILE 1：ORIGINAL
        # ====================================================

        if profile == "original":

            # ------------------------------------------------
            # 20260725 原版
            #
            # Rotation ±5°
            # Translation ±5%
            # Scale 0.9–1.1
            # Shear ±3°
            # ------------------------------------------------

            train_tf_list.append(

                transforms.RandomAffine(

                    degrees=5,

                    translate=(
                        0.05,
                        0.05
                    ),

                    scale=(
                        0.9,
                        1.1
                    ),

                    shear=3,

                    interpolation=
                    InterpolationMode.BILINEAR,

                    fill=255,

                )

            )


            # ------------------------------------------------
            # Brightness / Contrast
            # ------------------------------------------------

            train_tf_list.append(

                transforms.ColorJitter(

                    brightness=0.15,

                    contrast=0.15,

                )

            )



        # ====================================================
        # PROFILE 2：LIU 2026
        # ====================================================

        elif profile == "liu2026":

            # ------------------------------------------------
            # 保守型 ECG augmentation
            #
            # Rotation ±3°
            # Translation ±5%
            # ------------------------------------------------

            train_tf_list.append(

                transforms.RandomAffine(

                    degrees=
                    cfg[
                        "AUG_ROTATION_DEGREE"
                    ],

                    translate=
                    cfg[
                        "AUG_TRANSLATE"
                    ],

                    interpolation=
                    InterpolationMode.BILINEAR,

                    fill=255,

                )

            )



        # ====================================================
        # PROFILE 3：SAFE
        # PROFILE 4：REAL WORLD
        # ====================================================

        elif profile in (
            "safe",
            "real_world"
        ):

            # ------------------------------------------------
            # ECG-safe affine
            #
            # Rotation ±3°
            # Translation ±5%
            # Scale 0.95–1.05
            #
            # 目的：
            # 保留 P-QRS-T / ST / QT morphology
            # ------------------------------------------------

            train_tf_list.append(

                transforms.RandomAffine(

                    degrees=
                    cfg[
                        "AUG_ROTATION_DEGREE"
                    ],

                    translate=
                    cfg[
                        "AUG_TRANSLATE"
                    ],

                    scale=
                    cfg[
                        "AUG_SCALE"
                    ],

                    interpolation=
                    InterpolationMode.BILINEAR,

                    fill=255,

                )

            )


            # ------------------------------------------------
            # Brightness / Contrast
            # ------------------------------------------------

            train_tf_list.append(

                transforms.ColorJitter(

                    brightness=
                    cfg[
                        "AUG_BRIGHTNESS"
                    ],

                    contrast=
                    cfg[
                        "AUG_CONTRAST"
                    ],

                )

            )


            # =================================================
            # REAL WORLD
            # =================================================

            if profile == "real_world":


                # ---------------------------------------------
                # Perspective
                #
                # 模擬：
                # - 手機拍攝角度
                # - 掃描紙張角度
                # ---------------------------------------------

                train_tf_list.append(

                    transforms.RandomPerspective(

                        distortion_scale=
                        cfg[
                            "AUG_PERSPECTIVE_DISTORTION"
                        ],

                        p=
                        cfg[
                            "AUG_PERSPECTIVE_P"
                        ],

                        interpolation=
                        InterpolationMode.BILINEAR,

                        fill=255,

                    )

                )


                # ---------------------------------------------
                # Slight Gaussian Blur
                #
                # 模擬：
                # - 掃描模糊
                # - 手機失焦
                #
                # 機率與強度保持低
                # ---------------------------------------------

                train_tf_list.append(

                    transforms.RandomApply(

                        [

                            transforms.GaussianBlur(

                                kernel_size=
                                cfg[
                                    "AUG_BLUR_KERNEL_SIZE"
                                ],

                                sigma=
                                cfg[
                                    "AUG_BLUR_SIGMA"
                                ],

                            )

                        ],

                        p=
                        cfg[
                            "AUG_BLUR_P"
                        ],

                    )

                )


        # ====================================================
        # ECG Flip
        # ====================================================

        hflip_p = float(

            cfg.get(

                "AUG_HORIZONTAL_FLIP_P",

                0.0

            )

        )


        vflip_p = float(

            cfg.get(

                "AUG_VERTICAL_FLIP_P",

                0.0

            )

        )


        # ----------------------------------------------------
        # Horizontal Flip
        # ----------------------------------------------------

        if hflip_p > 0:

            print(

                "[WARNING] "
                "AUG_HORIZONTAL_FLIP_P > 0："
                "ECG 時間方向會被反轉，通常不建議。"

            )

            train_tf_list.append(

                transforms.RandomHorizontalFlip(

                    p=hflip_p

                )

            )


        # ----------------------------------------------------
        # Vertical Flip
        # ----------------------------------------------------

        if vflip_p > 0:

            print(

                "[WARNING] "
                "AUG_VERTICAL_FLIP_P > 0："
                "ECG 電位方向會被反轉，通常不建議。"

            )

            train_tf_list.append(

                transforms.RandomVerticalFlip(

                    p=vflip_p

                )

            )



    # ========================================================
    # ToTensor + Normalize
    # ========================================================

    train_tf_list.extend(

        [

            transforms.ToTensor(),

            transforms.Normalize(

                mean=
                cfg[
                    "NORMALIZE_MEAN"
                ],

                std=
                cfg[
                    "NORMALIZE_STD"
                ],

            ),

        ]

    )


    # ========================================================
    # Original Random Erasing
    # ========================================================
    #
    # 只有 original profile 使用。
    #
    # Safe / Liu / Real-world 不使用，
    # 避免擦掉 ECG waveform。
    #
    # ========================================================

    if (

        use_aug

        and

        profile == "original"

        and

        cfg.get(
            "AUG_RANDOM_ERASING_P",
            0.0
        ) > 0

    ):

        train_tf_list.append(

            transforms.RandomErasing(

                p=
                cfg[
                    "AUG_RANDOM_ERASING_P"
                ],

                scale=(
                    0.02,
                    0.08
                ),

                value=1.0,

            )

        )


    # ========================================================
    # Train Transform
    # ========================================================

    train_tf = transforms.Compose(

        train_tf_list

    )


    # ========================================================
    # Validation / Testing Transform
    # ========================================================
    #
    # Val / Test 永遠不做 augmentation
    #
    # 只做：
    #
    # Top Mask
    # Resize
    # ToTensor
    # Normalize
    #
    # ========================================================

    eval_tf = transforms.Compose(

        base_pre

        +

        [

            transforms.ToTensor(),

            transforms.Normalize(

                mean=
                cfg[
                    "NORMALIZE_MEAN"
                ],

                std=
                cfg[
                    "NORMALIZE_STD"
                ],

            ),

        ]

    )


    # ========================================================
    # Print augmentation profile
    # ========================================================

    _print_augmentation_profile(

        cfg

    )


    return (

        train_tf,

        eval_tf

    )



# ============================================================
# Augmentation Preview
# ============================================================

def visualize_augmentations(
    image_path: str,
    cfg: dict,
    n_examples: int = 5,
    output_dir: str = None
):

    """
    顯示：

    1. 原始 ECG
    2. Top mask 後 ECG
    3. N 張 augmentation 結果

    建議：
    每次修改 AUG_PROFILE 後先執行一次，
    確認 ECG waveform 沒有被過度扭曲。
    """

    import matplotlib.pyplot as plt


    # ========================================================
    # Denormalization
    # ========================================================

    def _denorm(
        img_tensor
    ):

        mean = np.asarray(
            cfg[
                "NORMALIZE_MEAN"
            ]
        )

        std = np.asarray(
            cfg[
                "NORMALIZE_STD"
            ]
        )


        img = (

            img_tensor
            .clone()
            .cpu()
            .numpy()
            .transpose(
                1,
                2,
                0
            )

        )


        img = (

            img
            *
            std
            +
            mean

        )


        return np.clip(

            img,

            0,

            1

        )


    # ========================================================
    # Build transform
    # ========================================================

    train_tf, _ = get_transforms(

        cfg

    )


    # ========================================================
    # Load ECG image
    # ========================================================

    raw_img = Image.open(

        image_path

    ).convert(

        "RGB"

    )


    # ========================================================
    # Preview top mask
    # ========================================================

    mask_top = MaskTopRegion(

        top_ratio=
        cfg[
            "MASK_TOP_RATIO"
        ],

        fill_color=(
            255,
            255,
            255
        )

    )


    masked_preview = (

        mask_top(
            raw_img
        )
        .resize(

            (
                cfg[
                    "IMG_SIZE"
                ],

                cfg[
                    "IMG_SIZE"
                ]

            )

        )

    )


    # ========================================================
    # Figure
    # ========================================================

    n_cols = (

        n_examples

        +

        2

    )


    fig, axes = plt.subplots(

        1,

        n_cols,

        figsize=(

            3 * n_cols,

            3.4

        )

    )


    # ========================================================
    # Original
    # ========================================================

    axes[0].imshow(

        raw_img

    )

    axes[0].set_title(

        "原圖\n(Original)",

        fontsize=10

    )

    axes[0].axis(

        "off"

    )


    # ========================================================
    # Masked
    # ========================================================

    axes[1].imshow(

        masked_preview

    )


    axes[1].set_title(

        f"遮罩後\n"
        f"(top "
        f"{int(cfg['MASK_TOP_RATIO'] * 100)}%)",

        fontsize=10

    )


    axes[1].axis(

        "off"

    )


    # ========================================================
    # Augmentation examples
    # ========================================================

    for i in range(

        n_examples

    ):

        aug_tensor = train_tf(

            raw_img

        )


        aug_img = _denorm(

            aug_tensor

        )


        axes[
            2 + i
        ].imshow(

            aug_img

        )


        axes[
            2 + i
        ].set_title(

            f"{cfg['AUG_PROFILE']} "
            f"#{i + 1}",

            fontsize=10

        )


        axes[
            2 + i
        ].axis(

            "off"

        )


    # ========================================================
    # Title
    # ========================================================

    plt.suptitle(

        f"ECG Data Augmentation Preview "
        f"— "
        f"{cfg['AUG_PROFILE']}",

        fontsize=13,

        fontweight="bold"

    )


    plt.tight_layout()


    # ========================================================
    # Save
    # ========================================================

    if output_dir:

        os.makedirs(

            output_dir,

            exist_ok=True

        )


        save_path = os.path.join(

            output_dir,

            f"augmentation_preview_"
            f"{cfg['AUG_PROFILE']}.png"

        )


        plt.savefig(

            save_path,

            dpi=150,

            bbox_inches="tight"

        )


        print(

            f"[preview] 儲存: "
            f"{save_path}"

        )


    plt.show()

    plt.close()


print("G")

G


In [8]:
# BLOCK H: 多模態 Dataset
# ============================================================
class PTBXLMultimodalDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform, age_scaler: AgeScaler,
                 label2idx: dict, use_age: bool = True, use_sex: bool = True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.age_scaler = age_scaler
        self.label2idx = label2idx
        self.use_age = use_age
        self.use_sex = use_sex

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["img_path"]).convert("RGB")
        image = self.transform(image)

        demo_values = []
        if self.use_age:
            age_norm = self.age_scaler.transform(np.array([row["age_clean"]]))[0]
            demo_values.append(age_norm)
        if self.use_sex:
            demo_values.append(float(row["sex_clean"]))

        if len(demo_values) > 0:
            demo_tensor = torch.tensor(demo_values, dtype=torch.float32)
        else:
            demo_tensor = torch.tensor([0.0], dtype=torch.float32)

        label = self.label2idx[row["label"]]
        return image, demo_tensor, label

print('H')



H


In [9]:
# BLOCK I: WeightedRandomSampler 建構 (處理類別不平衡)
# ============================================================
def build_weighted_sampler(labels: list) -> WeightedRandomSampler:
    class_counts = Counter(labels)
    num_samples = len(labels)
    class_weights = {c: num_samples / count for c, count in class_counts.items()}
    sample_weights = [class_weights[l] for l in labels]
    return WeightedRandomSampler(
        weights=sample_weights, num_samples=num_samples, replacement=True
    )
print('I')

I


In [10]:
# BLOCK J: DataLoader 建構
# ============================================================
def build_dataloaders(train_df, val_df, test_df, label2idx, cfg: dict):
    train_tf, eval_tf = get_transforms(cfg)
 
    age_scaler = AgeScaler().fit(train_df["age_clean"].values)
 
    train_ds = PTBXLMultimodalDataset(train_df, train_tf, age_scaler, label2idx,
                                       cfg["USE_AGE_MODALITY"], cfg["USE_SEX_MODALITY"])
    val_ds = PTBXLMultimodalDataset(val_df, eval_tf, age_scaler, label2idx,
                                     cfg["USE_AGE_MODALITY"], cfg["USE_SEX_MODALITY"])
    test_ds = PTBXLMultimodalDataset(test_df, eval_tf, age_scaler, label2idx,
                                      cfg["USE_AGE_MODALITY"], cfg["USE_SEX_MODALITY"])
 
    if cfg["USE_WEIGHTED_SAMPLER"]:
        train_labels_idx = [label2idx[l] for l in train_df["label"]]
        sampler = build_weighted_sampler(train_labels_idx)
        train_loader = DataLoader(train_ds, batch_size=cfg["BATCH_SIZE"], sampler=sampler,
                                   num_workers=cfg["NUM_WORKERS"])
    else:
        train_loader = DataLoader(train_ds, batch_size=cfg["BATCH_SIZE"], shuffle=True,
                                   num_workers=cfg["NUM_WORKERS"])
 
    val_loader = DataLoader(val_ds, batch_size=cfg["BATCH_SIZE"], shuffle=False,
                             num_workers=cfg["NUM_WORKERS"])
    test_loader = DataLoader(test_ds, batch_size=cfg["BATCH_SIZE"], shuffle=False,
                              num_workers=cfg["NUM_WORKERS"])
 
    return train_loader, val_loader, test_loader, age_scaler
print('J')

J


In [11]:
# BLOCK K: CBAM-ResNet18 + 多模態 Age/Sex Fusion
# ============================================================

class ChannelAttention(nn.Module):
    """
    CBAM - Channel Attention

    對每個 feature channel 計算重要性。
    使用：
      Global Average Pooling
      Global Max Pooling
      Shared MLP
      Sigmoid
    """

    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()

        hidden_channels = max(channels // reduction, 1)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.shared_mlp = nn.Sequential(
            nn.Conv2d(channels, hidden_channels, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, channels, kernel_size=1, bias=False),
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))

        attention = self.sigmoid(avg_out + max_out)

        return x * attention


class SpatialAttention(nn.Module):
    """
    CBAM - Spatial Attention

    沿 channel 維度計算：
      mean map
      max map

    再 concat 後使用 convolution 產生 spatial attention map。
    """

    def __init__(self, kernel_size: int = 7):
        super().__init__()

        if kernel_size not in (3, 7):
            raise ValueError(
                "CBAM_SPATIAL_KERNEL 建議使用 3 或 7。"
            )

        padding = kernel_size // 2

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size=kernel_size,
            padding=padding,
            bias=False,
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_map = torch.mean(
            x,
            dim=1,
            keepdim=True
        )

        max_map, _ = torch.max(
            x,
            dim=1,
            keepdim=True
        )

        spatial_input = torch.cat(
            [avg_map, max_map],
            dim=1
        )

        attention = self.sigmoid(
            self.conv(spatial_input)
        )

        return x * attention


class CBAM(nn.Module):
    """
    完整 CBAM：
        Channel Attention
        ↓
        Spatial Attention
    """

    def __init__(
        self,
        channels: int,
        reduction: int = 16,
        spatial_kernel: int = 7,
    ):
        super().__init__()

        self.channel = ChannelAttention(
            channels=channels,
            reduction=reduction,
        )

        self.spatial = SpatialAttention(
            kernel_size=spatial_kernel
        )

    def forward(self, x):
        x = self.channel(x)
        x = self.spatial(x)
        return x


class CBAMResNet18Encoder(nn.Module):
    """
    ImageNet-pretrained ResNet18 + CBAM。

    CBAM 放置方式：
        Stem
         ↓
        ResNet layer1 → CBAM1
         ↓
        ResNet layer2 → CBAM2
         ↓
        ResNet layer3 → CBAM3
         ↓
        ResNet layer4 → CBAM4
         ↓
        Global Average Pooling
         ↓
        512-D image feature

    注意：
    這是「標準 CBAM 概念 + ResNet18」的研究實作，
    不是聲稱逐行重現某篇論文作者的 private source code。
    """

    def __init__(
        self,
        pretrained: bool = True,
        reduction: int = 16,
        spatial_kernel: int = 7,
    ):
        super().__init__()

        # ----------------------------------------------------
        # ImageNet pretrained ResNet18
        # ----------------------------------------------------
        try:
            weights = (
                models.ResNet18_Weights.IMAGENET1K_V1
                if pretrained
                else None
            )

            base = models.resnet18(
                weights=weights
            )

        except AttributeError:
            # 舊版 torchvision 相容
            base = models.resnet18(
                pretrained=pretrained
            )

        # ----------------------------------------------------
        # ResNet stem
        # ----------------------------------------------------
        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool

        # ----------------------------------------------------
        # Residual stages
        # ----------------------------------------------------
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4

        # ----------------------------------------------------
        # CBAM after each ResNet stage
        # ----------------------------------------------------
        self.cbam1 = CBAM(
            channels=64,
            reduction=reduction,
            spatial_kernel=spatial_kernel,
        )

        self.cbam2 = CBAM(
            channels=128,
            reduction=reduction,
            spatial_kernel=spatial_kernel,
        )

        self.cbam3 = CBAM(
            channels=256,
            reduction=reduction,
            spatial_kernel=spatial_kernel,
        )

        self.cbam4 = CBAM(
            channels=512,
            reduction=reduction,
            spatial_kernel=spatial_kernel,
        )

        self.avgpool = base.avgpool

        # 與 timm encoder 介面保持一致
        self.num_features = 512

        # 原 ResNet fc 不使用
        del base

    def forward(self, x):
        # Stem
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        # Stage 1
        x = self.layer1(x)
        x = self.cbam1(x)

        # Stage 2
        x = self.layer2(x)
        x = self.cbam2(x)

        # Stage 3
        x = self.layer3(x)
        x = self.cbam3(x)

        # Stage 4
        x = self.layer4(x)
        x = self.cbam4(x)

        # Global Average Pooling
        x = self.avgpool(x)

        # [B, 512, 1, 1] -> [B, 512]
        x = torch.flatten(
            x,
            1
        )

        return x


class MultimodalECGNet(nn.Module):
    """
    ECG image encoder
        +
    Age/Sex demographic encoder
        ↓
    Feature Fusion
        ↓
    Classification Head
    """

    def __init__(
        self,
        backbone: str,
        pretrained: bool,
        num_classes: int,
        demo_embed_dim: int = 16,
        use_age: bool = True,
        use_sex: bool = True,
        drop_path_rate: float = 0.0,
    ):
        super().__init__()

        self.use_age = use_age
        self.use_sex = use_sex
        self.use_demo = use_age or use_sex

        # ====================================================
        # Image Encoder
        # ====================================================
        self.img_encoder = self._build_img_encoder(
            backbone=backbone,
            pretrained=pretrained,
            drop_path_rate=drop_path_rate,
        )

        img_feat_dim = self.img_encoder.num_features

        # ====================================================
        # Demographic Encoder
        # ====================================================
        demo_input_dim = (
            int(use_age)
            + int(use_sex)
        )

        if self.use_demo:
            self.demo_encoder = nn.Sequential(
                nn.Linear(
                    demo_input_dim,
                    demo_embed_dim
                ),
                nn.ReLU(),
                nn.Linear(
                    demo_embed_dim,
                    demo_embed_dim
                ),
                nn.ReLU(),
            )

            fusion_in_dim = (
                img_feat_dim
                + demo_embed_dim
            )

        else:
            self.demo_encoder = None
            fusion_in_dim = img_feat_dim

        # ====================================================
        # Classification Head
        # ====================================================
        self.classifier = nn.Sequential(
            nn.Linear(
                fusion_in_dim,
                128
            ),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(
                128,
                num_classes
            ),
        )

    @staticmethod
    def _build_img_encoder(
        backbone: str,
        pretrained: bool,
        drop_path_rate: float,
    ):
        # ----------------------------------------------------
        # CBAM-ResNet18
        # ----------------------------------------------------
        if backbone == "cbam_resnet18":
            print(
                "[model] 建立 CBAM-ResNet18 "
                f"(reduction={CONFIG['CBAM_REDUCTION']}, "
                f"spatial_kernel={CONFIG['CBAM_SPATIAL_KERNEL']})"
            )

            if drop_path_rate not in (0, 0.0, None):
                print(
                    "[warn] CBAM-ResNet18 不使用 DROP_PATH_RATE，"
                    "此設定已忽略。"
                )

            return CBAMResNet18Encoder(
                pretrained=pretrained,
                reduction=CONFIG["CBAM_REDUCTION"],
                spatial_kernel=CONFIG["CBAM_SPATIAL_KERNEL"],
            )

        # ----------------------------------------------------
        # 保留原本 timm backbone 支援
        # 之後切回 ConvNeXt / EfficientNet 不需重寫整個 Block
        # ----------------------------------------------------
        if (
            drop_path_rate
            and drop_path_rate > 0
        ):
            try:
                return timm.create_model(
                    backbone,
                    pretrained=pretrained,
                    num_classes=0,
                    drop_path_rate=drop_path_rate,
                )
            except TypeError:
                print(
                    f"[warn] backbone='{backbone}' "
                    "不支援 drop_path_rate，已忽略。"
                )

        return timm.create_model(
            backbone,
            pretrained=pretrained,
            num_classes=0,
        )

    def forward(
        self,
        image,
        demo,
    ):
        img_feat = self.img_encoder(
            image
        )

        if self.use_demo:
            demo_feat = self.demo_encoder(
                demo
            )

            fused = torch.cat(
                [img_feat, demo_feat],
                dim=1
            )
        else:
            fused = img_feat

        output = self.classifier(
            fused
        )

        return output

    def freeze_backbone(self):
        for p in self.img_encoder.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.img_encoder.parameters():
            p.requires_grad = True


print("K")

K


In [12]:
# BLOCK L: Loss function (class-weighted CrossEntropy 或 Focal Loss)
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


def build_criterion(train_labels_idx: list, num_classes: int, cfg: dict):
    counts = Counter(train_labels_idx)
    weights = torch.tensor(
        [len(train_labels_idx) / counts.get(c, 1) for c in range(num_classes)],
        dtype=torch.float32,
    ).to(DEVICE)
    # 常見做法：weight clipping，避免極端不平衡讓多數類別 recall 崩掉
    weights = torch.clamp(weights, min=0.5, max=5.0)

    if cfg["USE_FOCAL_LOSS"]:
        return FocalLoss(alpha=weights, gamma=cfg["FOCAL_GAMMA"])
    elif cfg["USE_CLASS_WEIGHTED_LOSS"]:
        return nn.CrossEntropyLoss(weight=weights)
    else:
        return nn.CrossEntropyLoss()

print('L')

L


In [13]:
# BLOCK M: Optimizer + Freeze/Unfreeze 排程
# ============================================================
def build_optimizer(model: MultimodalECGNet, cfg: dict):
    backbone_params = list(model.img_encoder.parameters())
    other_params = list(model.classifier.parameters())
    if model.use_demo:
        other_params += list(model.demo_encoder.parameters())

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": cfg["LR_BACKBONE"]},
        {"params": other_params, "lr": cfg["LR_HEAD"]},
    ], weight_decay=cfg["WEIGHT_DECAY"])
    return optimizer
print('M')

M


In [14]:
# BLOCK N: LR Scheduler
# ============================================================
def build_scheduler(optimizer, cfg: dict):
    if cfg["SCHEDULER_TYPE"] == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["EPOCHS"])
    else:  # "plateau" (預設)
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=5,           # 注意: 要小於EARLY_STOP_PATIENCE，否則同時觸發
            min_lr=1e-6,
        )


def step_scheduler(scheduler, cfg: dict, metric_value: float):
    """統一入口：依 SCHEDULER_TYPE 決定 step() 要不要傳指標值。
    Block R 只需呼叫這個函式，之後在 CONFIG 切換 scheduler 種類都不用改 Block R。"""
    if cfg["SCHEDULER_TYPE"] == "cosine":
        scheduler.step()
    else:
        scheduler.step(metric_value)

print('N')

N


In [15]:
# BLOCK O: Mixup (只作用於影像分支，年齡分支維持原值傳遞，預設關閉)
# ============================================================
def mixup_data(images, ages, labels, alpha: float):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = images.size(0)
    index = torch.randperm(batch_size).to(images.device)

    mixed_images = lam * images + (1 - lam) * images[index]
    # 年齡不做混合，保留原始年齡對應原始影像的邏輯較合理；若要混合可改成加權平均
    labels_a, labels_b = labels, labels[index]
    return mixed_images, ages, labels_a, labels_b, lam


def mixup_criterion(criterion, pred, labels_a, labels_b, lam):
    return lam * criterion(pred, labels_a) + (1 - lam) * criterion(pred, labels_b)


print('O')

O


In [16]:
# BLOCK P: 訓練 / 驗證迴圈
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion, cfg: dict):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
 
    for batch_idx, (images, demo, labels) in enumerate(loader):
        images, demo, labels = images.to(DEVICE), demo.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
 
        if cfg["USE_MIXUP"]:
            mixed_images, demo, labels_a, labels_b, lam = mixup_data(images, demo, labels, cfg["MIXUP_ALPHA"])
            outputs = model(mixed_images, demo)
            loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
        else:
            outputs = model(images, demo)
            loss = criterion(outputs, labels)
 
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
 
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += images.size(0)
 
        if cfg.get("VERBOSE_BATCH", False) and batch_idx % 50 == 0:
            print(f"  batch {batch_idx}/{len(loader)} loss={loss.item():.4f}")
 
    avg_loss = total_loss / len(loader.dataset)
    accuracy = total_correct / total_samples
    return avg_loss, accuracy
 
 
@torch.no_grad()
def evaluate(model, loader, criterion, idx2label: dict):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
 
    for images, demo, labels in loader:
        images, demo, labels = images.to(DEVICE), demo.to(DEVICE), labels.to(DEVICE)
        outputs = model(images, demo)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
 
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
 
    val_loss = total_loss / len(loader.dataset)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
 
    from sklearn.metrics import precision_score, recall_score, accuracy_score
    precision_macro = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall_macro = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    accuracy = accuracy_score(all_labels, all_preds)
 
    report = classification_report(
        all_labels, all_preds,
        target_names=[idx2label[i] for i in range(len(idx2label))],
        zero_division=0,
    )
    cm = confusion_matrix(all_labels, all_preds)
 
    return {
        "val_loss": val_loss,
        "val_f1_macro": f1_macro,
        "val_precision_macro": precision_macro,
        "val_recall_macro": recall_macro,
        "val_accuracy": accuracy,
        "report": report, "cm": cm,
        "all_labels": all_labels, "all_preds": all_preds,
    }
print('P')

P


In [17]:
# BLOCK Q: Grad-CAM (只作用於影像分支)
# ============================================================
class GradCAM:
    def __init__(self, model: MultimodalECGNet, target_layer_name: str):
        self.model = model
        self.gradients = None
        self.activations = None

        target_layer = dict([*model.img_encoder.named_modules()])[target_layer_name]
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, image_tensor, demo_tensor, class_idx=None):
        self.model.eval()
        image_tensor = image_tensor.unsqueeze(0).to(DEVICE)
        demo_tensor = demo_tensor.unsqueeze(0).to(DEVICE)

        output = self.model(image_tensor, demo_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, class_idx].backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=image_tensor.shape[2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

    # 注意: Grad-CAM 只反映影像分支的空間注意力，年齡/性別分支的貢獻無法用此方式視覺化，
    # 解讀結果時務必註明「此為影像模態的注意力，不代表人口學模態的影響程度」。


def denormalize_image(img_tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    """把經過 ImageNet Normalize 的 tensor 還原成可顯示的 [0,1] RGB 影像"""
    img = img_tensor.clone().cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(std) + np.array(mean)
    return np.clip(img, 0, 1)


def plot_gradcam_single(image_tensor, cam, pred_label: str, true_label: str, save_path: str):
    """單筆樣本的 Grad-CAM 三合一圖: 原圖 / 熱力圖 / 疊圖"""
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm

    img = denormalize_image(image_tensor)
    heatmap_rgb = cm.jet(cam)[..., :3]
    overlay = np.clip(0.55 * img + 0.45 * heatmap_rgb, 0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img)
    axes[0].set_title("Original ECG Image")
    axes[0].axis("off")

    axes[1].imshow(cam, cmap="jet")
    axes[1].set_title("Grad-CAM Heatmap\n(模型重點關注區域)")
    axes[1].axis("off")

    correct = "✓" if pred_label == true_label else "✗"
    axes[2].imshow(overlay)
    axes[2].set_title(f"Overlay  [{correct}]  pred={pred_label} / true={true_label}")
    axes[2].axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


def generate_gradcam_examples(model, test_loader, idx2label: dict, target_layer: str, output_dir: str,
                               n_per_class: int = 1):
    """每個類別各挑 n_per_class 筆 test 樣本，產出 Grad-CAM 視覺化，方便逐類別檢視模型關注區域"""
    gradcam = GradCAM(model, target_layer)
    dataset = test_loader.dataset

    class_to_indices = {i: [] for i in range(len(idx2label))}
    for idx in range(len(dataset)):
        label = dataset.df.iloc[idx]["label"]
        label_idx = [k for k, v in idx2label.items() if v == label][0]
        class_to_indices[label_idx].append(idx)

    saved_paths = []
    for class_idx, indices in class_to_indices.items():
        class_name = idx2label[class_idx]
        chosen = indices[:n_per_class]
        for i, sample_idx in enumerate(chosen):
            image_tensor, demo_tensor, true_idx = dataset[sample_idx]
            cam, pred_idx = gradcam.generate(image_tensor, demo_tensor)
            save_path = os.path.join(output_dir, f"gradcam_{class_name}_{i}.png")
            plot_gradcam_single(image_tensor, cam, idx2label[pred_idx], idx2label[true_idx], save_path)
            saved_paths.append(save_path)
            print(f"[gradcam] 儲存: {save_path}")

    return saved_paths
print('Q')

Q


In [18]:
# BLOCK R0: Kaggle Checkpoint / Resume（12 小時中斷續跑）
# ============================================================
def _checkpoint_names(cfg: dict):
    exp = cfg["EXPERIMENT_NAME"]
    return {
        "last": f"{exp}_last.pth",
        "best": f"{exp}_best.pth",
        "history": f"{exp}_history.csv",
        "summary": f"{exp}_checkpoint_info.json",
    }


def _resume_signature(cfg: dict):
    """
    只放「不能在續跑中途改掉」的核心設定。
    若不同，預設拒絕 resume，避免載錯 checkpoint。
    """
    return {
        "experiment_name": cfg["EXPERIMENT_NAME"],
        "backbone": cfg["BACKBONE"],
        "num_classes": cfg["NUM_CLASSES"],
        "img_size": cfg["IMG_SIZE"],
        "use_age": cfg["USE_AGE_MODALITY"],
        "use_sex": cfg["USE_SEX_MODALITY"],
        "demo_embed_dim": cfg["DEMO_EMBED_DIM"],
        "drop_path_rate": cfg.get("DROP_PATH_RATE", 0.0),
        "cbam_reduction": cfg.get("CBAM_REDUCTION", None),
        "cbam_spatial_kernel": cfg.get("CBAM_SPATIAL_KERNEL", None),
        "single_label_only": cfg["SINGLE_LABEL_ONLY"],
        "img_rate": cfg["IMG_RATE"],
        "max_img_per_record": cfg["MAX_IMG_PER_RECORD"],
    }


def _find_exact_file(filename: str, roots: list):
    """
    在 /kaggle/working 與 /kaggle/input 中遞迴搜尋指定檔名。
    若找到多個，優先選 mtime 最新者。
    """
    candidates = []
    for root in roots:
        if not root or not os.path.exists(root):
            continue
        pattern = os.path.join(root, "**", filename)
        candidates.extend(glob.glob(pattern, recursive=True))

    if not candidates:
        return None

    candidates = sorted(
        set(candidates),
        key=lambda p: os.path.getmtime(p),
        reverse=True,
    )
    return candidates[0]


def resolve_resume_checkpoint(cfg: dict):
    if not cfg.get("RESUME_TRAINING", False):
        return None

    explicit = cfg.get("RESUME_CHECKPOINT")
    if explicit:
        if os.path.exists(explicit):
            print(f"[resume] 使用指定 checkpoint: {explicit}")
            return explicit
        raise FileNotFoundError(
            f"RESUME_CHECKPOINT 指定的檔案不存在：{explicit}"
        )

    names = _checkpoint_names(cfg)
    found = _find_exact_file(
        names["last"],
        cfg.get("RESUME_SEARCH_ROOTS", []),
    )

    if found:
        print(f"[resume] 自動找到 last checkpoint: {found}")
    else:
        print("[resume] 沒找到舊 checkpoint，將從 epoch 1 開始。")
    return found


def _atomic_torch_save(obj, path: str):
    """
    先寫 .tmp，再 os.replace。
    避免 Kaggle / Kernel 在寫檔途中中斷造成 checkpoint 半毀。
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp_path = path + ".tmp"
    torch.save(obj, tmp_path)
    os.replace(tmp_path, path)


def _save_history_csv(history: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    pd.DataFrame(history).to_csv(path, index=False)


def _capture_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def _restore_rng_state(state):
    if not state:
        return
    try:
        random.setstate(state["python"])
        np.random.set_state(state["numpy"])
        torch.set_rng_state(state["torch"])
        if torch.cuda.is_available() and "cuda" in state:
            torch.cuda.set_rng_state_all(state["cuda"])
        print("[resume] RNG state 已恢復。")
    except Exception as e:
        print(f"[warn] RNG state 恢復失敗，將繼續訓練：{e}")


def _optimizer_to_device(optimizer, device):
    """torch.load(map_location='cpu') 後，把 optimizer state tensor 搬回 GPU。"""
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)


def save_training_checkpoint(
    cfg: dict,
    epoch_completed: int,
    model,
    optimizer,
    scheduler,
    best_metric: float,
    best_epoch,
    patience_counter: int,
    history: dict,
    label2idx: dict,
):
    names = _checkpoint_names(cfg)
    ckpt_dir = cfg["CHECKPOINT_DIR"]
    last_path = os.path.join(ckpt_dir, names["last"])
    history_path = os.path.join(ckpt_dir, names["history"])
    info_path = os.path.join(ckpt_dir, names["summary"])

    payload = {
        "format_version": 2,
        "experiment_name": cfg["EXPERIMENT_NAME"],
        "resume_signature": _resume_signature(cfg),
        "epoch": epoch_completed,  # 1-based：已完成到第幾個 epoch
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "best_metric": float(best_metric),
        "best_epoch": best_epoch,
        "patience_counter": int(patience_counter),
        "history": history,
        "label2idx": label2idx,
        "rng_state": _capture_rng_state(),
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

    _atomic_torch_save(payload, last_path)

    if cfg.get("SAVE_HISTORY_EVERY_EPOCH", True):
        _save_history_csv(history, history_path)

    # 小型 JSON，讓你不用載入 300MB checkpoint 就知道存到哪個 epoch
    info = {
        "experiment_name": cfg["EXPERIMENT_NAME"],
        "epoch_completed": epoch_completed,
        "best_epoch": best_epoch,
        "best_metric": float(best_metric),
        "early_stop_patience_counter": int(patience_counter),
        "last_checkpoint": names["last"],
        "best_checkpoint": names["best"],
        "saved_at": payload["saved_at"],
    }
    with open(info_path, "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)

    print(f"  → 💾 Last checkpoint 已保存：epoch {epoch_completed}")
    print(f"     {last_path}")


def save_best_checkpoint(
    cfg: dict,
    epoch: int,
    model,
    metric_value: float,
    label2idx: dict,
):
    names = _checkpoint_names(cfg)
    best_path = os.path.join(cfg["CHECKPOINT_DIR"], names["best"])

    payload = {
        "format_version": 2,
        "experiment_name": cfg["EXPERIMENT_NAME"],
        "resume_signature": _resume_signature(cfg),
        "epoch": epoch,
        "metric": float(metric_value),
        "model_state_dict": model.state_dict(),
        "label2idx": label2idx,
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    _atomic_torch_save(payload, best_path)
    print(f"  → ⭐ Best checkpoint 已寫入硬碟：{best_path}")



def _torch_load_full(path: str, map_location="cpu"):
    """
    PyTorch 2.6+ 對 torch.load 的 weights_only 預設行為有變更。
    checkpoint 含 optimizer / RNG / numpy state，因此必須完整載入。
    同時相容較舊 PyTorch（沒有 weights_only 參數）。
    """
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=map_location,
        )


def load_resume_state(
    checkpoint_path: str,
    cfg: dict,
    model,
    optimizer,
    scheduler,
    label2idx: dict,
):
    """
    回傳：
      start_epoch        : 0-based，for range(start_epoch, EPOCHS)
      best_metric
      best_epoch
      patience_counter
      history
    """
    if checkpoint_path is None:
        history = {
            "epoch": [],
            "train_loss": [],
            "val_loss": [],
            "train_accuracy": [],
            "val_f1_macro": [],
            "val_precision_macro": [],
            "val_recall_macro": [],
            "val_accuracy": [],
        }
        return 0, -np.inf, None, 0, history

    print(f"\n[resume] 載入 checkpoint：{checkpoint_path}")
    ckpt = _torch_load_full(checkpoint_path, map_location="cpu")

    saved_sig = ckpt.get("resume_signature", {})
    current_sig = _resume_signature(cfg)

    if saved_sig != current_sig:
        message = (
            "Checkpoint 與目前 CONFIG 不相容。\\n"
            f"Saved : {saved_sig}\\n"
            f"Current: {current_sig}"
        )
        if cfg.get("STRICT_RESUME_CONFIG", True):
            raise ValueError(message)
        print("[warn]", message)

    saved_label2idx = ckpt.get("label2idx")
    if saved_label2idx is not None and saved_label2idx != label2idx:
        raise ValueError(
            "label2idx 與 checkpoint 不一致，為避免類別 mapping 錯位已停止 resume。\\n"
            f"Saved={saved_label2idx}, Current={label2idx}"
        )

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    _optimizer_to_device(optimizer, DEVICE)

    if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    _restore_rng_state(ckpt.get("rng_state"))

    completed_epoch = int(ckpt["epoch"])
    start_epoch = completed_epoch  # epoch 是 1-based，因此下一輪 index 正好等於 completed_epoch

    best_metric = float(ckpt.get("best_metric", -np.inf))
    best_epoch = ckpt.get("best_epoch")
    patience_counter = int(ckpt.get("patience_counter", 0))
    history = ckpt.get("history", {})

    print("=" * 70)
    print("RESUME SUCCESS")
    print("=" * 70)
    print(f"已完成到 epoch       : {completed_epoch}")
    print(f"下一個 epoch         : {start_epoch + 1}")
    print(f"目標總 epochs        : {cfg['EPOCHS']}")
    print(f"歷史最佳 metric      : {best_metric:.6f}")
    print(f"歷史最佳 epoch       : {best_epoch}")
    print(f"Early-stop counter   : {patience_counter}/{cfg['EARLY_STOP_PATIENCE']}")
    print("=" * 70)

    return start_epoch, best_metric, best_epoch, patience_counter, history


def copy_resume_best_checkpoint_to_working(resume_checkpoint_path: str, cfg: dict):
    """
    新 Kaggle session 時：
    last checkpoint 在 /kaggle/input/...，
    對應的 best checkpoint 通常也在同一份 Input。
    將 best copy 到 /kaggle/working/checkpoints，方便訓練結束載入。
    """
    if resume_checkpoint_path is None:
        return

    names = _checkpoint_names(cfg)
    local_best = os.path.join(cfg["CHECKPOINT_DIR"], names["best"])

    if os.path.exists(local_best):
        return

    same_dir_best = os.path.join(os.path.dirname(resume_checkpoint_path), names["best"])
    source_best = same_dir_best if os.path.exists(same_dir_best) else _find_exact_file(
        names["best"],
        cfg.get("RESUME_SEARCH_ROOTS", []),
    )

    if source_best:
        import shutil
        shutil.copy2(source_best, local_best)
        print(f"[resume] 已將歷史 best checkpoint 複製到 working：{local_best}")
    else:
        print("[warn] 找到 last checkpoint，但沒找到舊 best checkpoint。")
        print("       訓練仍可續跑；若後續沒有刷新最佳 F1，最終會改用 last model。")


def load_best_weights_or_fallback(model, cfg: dict):
    names = _checkpoint_names(cfg)
    best_path = os.path.join(cfg["CHECKPOINT_DIR"], names["best"])
    last_path = os.path.join(cfg["CHECKPOINT_DIR"], names["last"])

    if os.path.exists(best_path):
        ckpt = _torch_load_full(best_path, map_location="cpu")
        model.load_state_dict(ckpt["model_state_dict"])
        print(f"[final] 已載入最佳 checkpoint：epoch {ckpt.get('epoch')}")
        return ckpt.get("epoch")

    if os.path.exists(last_path):
        ckpt = _torch_load_full(last_path, map_location="cpu")
        model.load_state_dict(ckpt["model_state_dict"])
        print("[warn] best checkpoint 不存在，改載入 last checkpoint。")
        return ckpt.get("epoch")

    raise FileNotFoundError("找不到 best / last checkpoint，無法進行最終 test。")


print("R0")

R0


In [19]:
# BLOCK R: 主流程（支援 Kaggle 12 小時 checkpoint / resume）
# ============================================================

def main(cfg: dict):

    # 記錄本次 Kaggle Session 開始時間
    session_start_time = (
        time.time()
    )

    print(
        f"Device: {DEVICE}"
    )

    # ========================================================
    # 0. 先找 Resume Checkpoint
    # ========================================================

    resume_checkpoint = (
        resolve_resume_checkpoint(
            cfg
        )
    )

    # ========================================================
    # 1. 資料準備
    # ========================================================

    meta = load_ptbxl_metadata(

        cfg["META_CSV"],

        cfg["SCP_CSV"],

        cfg["SINGLE_LABEL_ONLY"],
    )

    meta = clean_age(

        meta,

        cfg["AGE_CLIP_MAX"]
    )

    meta = clean_sex(
        meta
    )

    long_df = build_image_index(

        meta,

        cfg["IMG_ROOT"],

        cfg["IMG_RATE"],

        cfg["MAX_IMG_PER_RECORD"],
    )

    print(
        f"總影像數: {len(long_df)}, "
        f"類別分布:\n"
        f"{long_df['label'].value_counts()}"
    )

    # ========================================================
    # Label Mapping
    # ========================================================

    labels_sorted = sorted(

        long_df[
            "label"
        ].unique()
    )

    label2idx = {

        label: i

        for i, label
        in enumerate(
            labels_sorted
        )
    }

    idx2label = {

        i: label

        for label, i
        in label2idx.items()
    }

    # ========================================================
    # Patient-level Split
    # ========================================================

    train_df, val_df, test_df = (

        patient_level_split(

            long_df,

            cfg["SEED"],
        )
    )

    print(

        "train/val/test = "

        f"{len(train_df)}/"

        f"{len(val_df)}/"

        f"{len(test_df)}"
    )

    # ========================================================
    # DataLoader
    # ========================================================

    (
        train_loader,
        val_loader,
        test_loader,
        age_scaler
    ) = build_dataloaders(

        train_df,

        val_df,

        test_df,

        label2idx,

        cfg,
    )

    # ========================================================
    # 2. Model / Loss / Optimizer / Scheduler
    # ========================================================

    # 如果已經有完整 checkpoint，
    # 不需要重新下載 ImageNet pretrained weights。
    #
    # checkpoint 之後會完整覆蓋 model weights。

    use_pretrained_for_build = (

        cfg["PRETRAINED"]

        if resume_checkpoint is None

        else False
    )

    if (
        resume_checkpoint is not None
        and cfg["PRETRAINED"]
    ):

        print(

            "[resume] 已有完整 "
            "model checkpoint，"

            "本次建模先 "
            "pretrained=False，"

            "再載入 checkpoint，"

            "避免重複下載 "
            "pretrained weights。"
        )

    # ========================================================
    # Model
    # ========================================================

    model = MultimodalECGNet(

        backbone=
            cfg["BACKBONE"],

        pretrained=
            use_pretrained_for_build,

        num_classes=
            cfg["NUM_CLASSES"],

        demo_embed_dim=
            cfg["DEMO_EMBED_DIM"],

        use_age=
            cfg["USE_AGE_MODALITY"],

        use_sex=
            cfg["USE_SEX_MODALITY"],

        drop_path_rate=
            cfg.get(
                "DROP_PATH_RATE",
                0.0
            ),

    ).to(
        DEVICE
    )

    # ========================================================
    # Train Labels
    # ========================================================

    train_labels_idx = [

        label2idx[label]

        for label
        in train_df["label"]
    ]

    # ========================================================
    # Criterion
    # ========================================================

    criterion = build_criterion(

        train_labels_idx,

        cfg["NUM_CLASSES"],

        cfg,
    )

    # ========================================================
    # Optimizer
    # ========================================================

    optimizer = build_optimizer(

        model,

        cfg
    )

    # ========================================================
    # Scheduler
    # ========================================================

    scheduler = build_scheduler(

        optimizer,

        cfg
    )

    # ========================================================
    # 3. 載入 checkpoint
    # ========================================================

    copy_resume_best_checkpoint_to_working(

        resume_checkpoint,

        cfg,
    )

    (
        start_epoch,

        best_metric,

        best_epoch,

        patience_counter,

        history,

    ) = load_resume_state(

        resume_checkpoint,

        cfg,

        model,

        optimizer,

        scheduler,

        label2idx,
    )

    # ========================================================
    # History 欄位防呆
    # ========================================================

    history_keys = [

        "epoch",

        "train_loss",

        "val_loss",

        "train_accuracy",

        "val_f1_macro",

        "val_precision_macro",

        "val_recall_macro",

        "val_accuracy",
    ]

    for key in history_keys:

        history.setdefault(
            key,
            []
        )

    # ========================================================
    # 已完成全部 epoch
    # ========================================================

    if (
        start_epoch
        >= cfg["EPOCHS"]
    ):

        print(

            "[resume] checkpoint "
            f"已完成 {start_epoch} epochs，"

            f"目前 EPOCHS="
            f"{cfg['EPOCHS']}，"

            "不需要再訓練。"
        )

    # ========================================================
    # Learning Rate 顯示
    # ========================================================

    def _fmt_lr(x):

        s = f"{x:.1e}"

        mantissa, exp = (
            s.split("e")
        )

        mantissa = (
            mantissa
            .rstrip("0")
            .rstrip(".")
        )

        exp = (
            exp
            .replace("+0", "+")
            .replace("-0", "-")
            .replace("+", "")
        )

        return (
            f"{mantissa}e{exp}"
        )

    backbone_name = (
        cfg["BACKBONE"]
    )

    print(
        "\n開始 / 接續訓練 "
        f"{backbone_name}"
    )

    print(
        "=" * 70
    )

    print(
        f"Start epoch : "
        f"{start_epoch + 1}"
    )

    print(
        f"Target epoch: "
        f"{cfg['EPOCHS']}"
    )

    # ========================================================
    # Freeze 狀態顯示
    # ========================================================

    if (
        start_epoch
        < cfg["FREEZE_EPOCHS"]
    ):

        print(

            "Backbone 仍在 "
            "freeze 階段；"

            "將於 epoch "
            f"{cfg['FREEZE_EPOCHS'] + 1} "

            "開始完整 fine-tune。"
        )

    else:

        print(

            "Resume 時已超過 "
            "freeze 階段，"

            "backbone 將直接 "
            "維持解凍。"
        )

    # ========================================================
    # 4. Training Loop
    # ========================================================

    early_stopped = False

    for epoch in range(

        start_epoch,

        cfg["EPOCHS"]
    ):

        epoch_start = (
            time.time()
        )

        # ====================================================
        # Freeze / Unfreeze
        # ====================================================

        if (
            epoch
            < cfg["FREEZE_EPOCHS"]
        ):

            model.freeze_backbone()

        else:

            if (
                epoch
                == cfg["FREEZE_EPOCHS"]
            ):

                print(

                    "\n🔥 "
                    f"[{backbone_name}] "
                    f"Epoch {epoch + 1}："
                    "解凍 backbone"
                )

                print(

                    "  分層學習率："

                    "分類頭 "
                    f"{_fmt_lr(cfg['LR_HEAD'])} "

                    "/ backbone "
                    f"{_fmt_lr(cfg['LR_BACKBONE'])}"
                )

            model.unfreeze_backbone()

        # ====================================================
        # Train
        # ====================================================

        train_loss, train_acc = (

            train_one_epoch(

                model,

                train_loader,

                optimizer,

                criterion,

                cfg,
            )
        )

        # ====================================================
        # Validation
        # ====================================================

        val_metrics = evaluate(

            model,

            val_loader,

            criterion,

            idx2label,
        )

        # ====================================================
        # Scheduler Step
        # ====================================================

        step_scheduler(

            scheduler,

            cfg,

            val_metrics[
                "val_f1_macro"
            ],
        )

        epoch_time = (

            time.time()
            - epoch_start
        )

        # ====================================================
        # Epoch Log
        # ====================================================

        print(

            f"[{backbone_name}] "

            f"Epoch "
            f"{epoch + 1:02d}/"
            f"{cfg['EPOCHS']} | "

            f"Loss: "
            f"{train_loss:.4f}/"
            f"{val_metrics['val_loss']:.4f} | "

            f"Acc: "
            f"{train_acc:.4f}/"
            f"{val_metrics['val_accuracy']:.4f} | "

            f"Prec: "
            f"{val_metrics['val_precision_macro']:.4f} | "

            f"Rec: "
            f"{val_metrics['val_recall_macro']:.4f} | "

            f"F1: "
            f"{val_metrics['val_f1_macro']:.4f} | "

            f"Time: "
            f"{epoch_time:.1f}s"
        )

        # ====================================================
        # History
        # ====================================================

        history["epoch"].append(

            epoch + 1
        )

        history["train_loss"].append(

            train_loss
        )

        history["train_accuracy"].append(

            train_acc
        )

        history["val_loss"].append(

            val_metrics[
                "val_loss"
            ]
        )

        history["val_f1_macro"].append(

            val_metrics[
                "val_f1_macro"
            ]
        )

        history[
            "val_precision_macro"
        ].append(

            val_metrics[
                "val_precision_macro"
            ]
        )

        history[
            "val_recall_macro"
        ].append(

            val_metrics[
                "val_recall_macro"
            ]
        )

        history[
            "val_accuracy"
        ].append(

            val_metrics[
                "val_accuracy"
            ]
        )

        current_metric = (

            val_metrics[
                cfg[
                    "EARLY_STOP_METRIC"
                ]
            ]
        )

        # ====================================================
        # 4-1 BEST CHECKPOINT
        # ====================================================

        if (
            current_metric
            > best_metric
        ):

            best_metric = (
                current_metric
            )

            best_epoch = (
                epoch + 1
            )

            patience_counter = 0

            save_best_checkpoint(

                cfg=cfg,

                epoch=
                    epoch + 1,

                model=model,

                metric_value=
                    current_metric,

                label2idx=
                    label2idx,
            )

            print(

                "  → ⭐ 新最佳 "

                f"{cfg['EARLY_STOP_METRIC']}: "

                f"{best_metric:.4f}"
            )

        else:

            patience_counter += 1

            print(

                "  → 未進步 "

                f"({patience_counter}/"
                f"{cfg['EARLY_STOP_PATIENCE']})"
            )

        # ====================================================
        # 4-2 LAST CHECKPOINT
        # ====================================================

        should_save_last = (

            (
                (epoch + 1)
                %
                cfg[
                    "SAVE_CHECKPOINT_EVERY"
                ]
                == 0
            )

            or

            (
                (epoch + 1)
                == cfg["EPOCHS"]
            )
        )

        if should_save_last:

            save_training_checkpoint(

                cfg=cfg,

                epoch_completed=
                    epoch + 1,

                model=model,

                optimizer=optimizer,

                scheduler=scheduler,

                best_metric=
                    best_metric,

                best_epoch=
                    best_epoch,

                patience_counter=
                    patience_counter,

                history=
                    history,

                label2idx=
                    label2idx,
            )

        # ====================================================
        # 4-3 EARLY STOPPING
        # ====================================================

        if (
            patience_counter
            >= cfg["EARLY_STOP_PATIENCE"]
        ):

            print(

                "\nEarly stopping "
                f"at epoch {epoch + 1} "

                "(best "
                f"{cfg['EARLY_STOP_METRIC']}="
                f"{best_metric:.4f}, "

                f"best epoch="
                f"{best_epoch})"
            )

            early_stopped = True

            break

        # ====================================================
        # 4-4 KAGGLE SAFE STOP
        # ====================================================

        if cfg.get(

            "GRACEFUL_STOP_BEFORE_KAGGLE_LIMIT",

            True
        ):

            elapsed_hours = (

                time.time()
                - session_start_time

            ) / 3600.0

            soft_limit = float(

                cfg.get(

                    "KAGGLE_SESSION_SOFT_LIMIT_HOURS",

                    10.5
                )
            )

            # 預估下一個 epoch
            predicted_next_hours = (

                epoch_time
                / 3600.0
                * 1.5
            )

            if (

                (epoch + 1)
                < cfg["EPOCHS"]

                and

                elapsed_hours
                + predicted_next_hours
                >= soft_limit
            ):

                print(
                    "\n"
                    + "=" * 70
                )

                print(
                    "KAGGLE SAFE STOP"
                )

                print(
                    "=" * 70
                )

                print(

                    "本 session 已執行約 "

                    f"{elapsed_hours:.2f} 小時。"
                )

                print(

                    "為避免接近 Kaggle "
                    "12 小時硬切，"

                    "已在 epoch "
                    f"{epoch + 1} "

                    "完整 checkpoint "
                    "後主動停止。"
                )

                print(

                    "請讓本次 "
                    "Save & Run All "

                    "正常完成，"

                    "下一個 Kaggle session "

                    "掛入本次 output 後 "

                    "Run All，"

                    "程式會從 epoch "

                    f"{epoch + 2} "

                    "自動續跑。"
                )

                print(
                    "=" * 70
                )

                break

    # ========================================================
    # 5. 載入 Best Weights
    # ========================================================

    final_best_epoch = (

        load_best_weights_or_fallback(

            model,

            cfg,
        )
    )

    model = (
        model.to(
            DEVICE
        )
    )

    # ========================================================
    # 6. Test Set
    # ========================================================

    test_metrics = evaluate(

        model,

        test_loader,

        criterion,

        idx2label,
    )

    print(
        "\n=== Test Set 結果 ==="
    )

    print(
        test_metrics[
            "report"
        ]
    )

    print(

        "Confusion Matrix:\n",

        test_metrics[
            "cm"
        ]
    )

    print()

    print_overall_summary(

        cfg["BACKBONE"],

        test_metrics,
    )

    # ========================================================
    # 額外儲存純模型 State Dict
    # ========================================================

    final_model_path = os.path.join(

        cfg["OUTPUT_DIR"],

        (
            f"{cfg['EXPERIMENT_NAME']}"
            "_best_model_state_dict.pth"
        )
    )

    torch.save(

        model.state_dict(),

        final_model_path
    )

    print(

        "[final] model state_dict："

        f"{final_model_path}"
    )

    # ========================================================
    # 7. 圖表
    # ========================================================

    class_names = [

        idx2label[i]

        for i
        in range(
            len(idx2label)
        )
    ]

    generate_all_plots(

        history=history,

        freeze_epochs=
            cfg["FREEZE_EPOCHS"],

        best_epoch=
            final_best_epoch,

        test_metrics=
            test_metrics,

        all_labels=
            test_metrics[
                "all_labels"
            ],

        all_preds=
            test_metrics[
                "all_preds"
            ],

        class_names=
            class_names,

        output_dir=
            cfg["OUTPUT_DIR"],
    )

    # ========================================================
    # 8. Grad-CAM
    # ========================================================

    if cfg[
        "USE_GRADCAM"
    ]:

        generate_gradcam_examples(

            model=model,

            test_loader=
                test_loader,

            idx2label=
                idx2label,

            target_layer=
                cfg[
                    "GRADCAM_TARGET_LAYER"
                ],

            output_dir=
                cfg["OUTPUT_DIR"],

            n_per_class=1,
        )

    # ========================================================
    # Finish
    # ========================================================

    print(
        "\n"
        + "=" * 70
    )

    print(
        "TRAINING FINISHED"
    )

    print(
        "=" * 70
    )

    print(
        "Experiment :",
        cfg["EXPERIMENT_NAME"]
    )

    print(
        "Best epoch :",
        final_best_epoch
    )

    print(
        "Best metric:",
        best_metric
    )

    print(
        "Checkpoint :",
        cfg["CHECKPOINT_DIR"]
    )

    print(
        "Outputs    :",
        cfg["OUTPUT_DIR"]
    )

    print(
        "=" * 70
    )

    return (
        model,
        test_metrics
    )


print("R")

R


In [20]:
# BLOCK S: 視覺化 (訓練曲線 / 混淆矩陣 / 各類別指標)
# ============================================================
def setup_cjk_font():
    """設定中文字型，避免 matplotlib CJK 缺字警告。找不到就靜默回退英文。"""
    import matplotlib
    from matplotlib import font_manager
    candidates = [
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
    ]
    for path in candidates:
        if os.path.exists(path):
            font_manager.fontManager.addfont(path)
            name = font_manager.FontProperties(fname=path).get_name()
            matplotlib.rcParams["font.family"] = name
            matplotlib.rcParams["axes.unicode_minus"] = False
            return True
    return False
 
 
def plot_training_curves(history: dict, freeze_epochs: int, best_epoch: int, output_dir: str):
    import matplotlib.pyplot as plt
    epochs = history["epoch"]
 
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
    axes[0].plot(epochs, history["train_loss"], marker="o", label="Train Loss", color="#E2B08C")
    axes[0].plot(epochs, history["val_loss"], marker="s", label="Val Loss", color="#8C9EE2")
    axes[0].axvline(x=freeze_epochs, color="gray", linestyle="--", alpha=0.6,
                     label=f"Backbone unfreeze (epoch {freeze_epochs})")
    axes[0].axvline(x=best_epoch, color="green", linestyle=":", alpha=0.8,
                     label=f"Best checkpoint (epoch {best_epoch})")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training / Validation Loss")
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.3)
 
    axes[1].plot(epochs, history["val_f1_macro"], marker="D", color="#C97B63")
    axes[1].axvline(x=freeze_epochs, color="gray", linestyle="--", alpha=0.6)
    axes[1].axvline(x=best_epoch, color="green", linestyle=":", alpha=0.8)
    best_idx = epochs.index(best_epoch)
    axes[1].scatter([best_epoch], [history["val_f1_macro"][best_idx]], color="green", s=100, zorder=5)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Val F1 (macro)")
    axes[1].set_title("Validation F1-macro")
    axes[1].grid(alpha=0.3)
 
    plt.tight_layout()
    path = os.path.join(output_dir, "01_training_curves.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {path}")
 
 
def plot_confusion_matrix(cm: np.ndarray, class_names: list, output_dir: str):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
 
    im0 = axes[0].imshow(cm, cmap="Oranges")
    axes[0].set_xticks(range(len(class_names)))
    axes[0].set_yticks(range(len(class_names)))
    axes[0].set_xticklabels(class_names)
    axes[0].set_yticklabels(class_names)
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")
    axes[0].set_title("Confusion Matrix (count)")
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            val = cm[i, j]
            color = "white" if val > cm.max() * 0.5 else "black"
            axes[0].text(j, i, str(val), ha="center", va="center", color=color, fontsize=10)
    plt.colorbar(im0, ax=axes[0], fraction=0.046)
 
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
    im1 = axes[1].imshow(cm_norm, cmap="Oranges", vmin=0, vmax=1)
    axes[1].set_xticks(range(len(class_names)))
    axes[1].set_yticks(range(len(class_names)))
    axes[1].set_xticklabels(class_names)
    axes[1].set_yticklabels(class_names)
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("True")
    axes[1].set_title("Confusion Matrix (row-normalized)")
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            val = cm_norm[i, j]
            color = "white" if val > 0.5 else "black"
            axes[1].text(j, i, f"{val:.0%}", ha="center", va="center", color=color, fontsize=10)
    plt.colorbar(im1, ax=axes[1], fraction=0.046)
 
    plt.tight_layout()
    path = os.path.join(output_dir, "02_confusion_matrix.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {path}")
 
 
def plot_per_class_metrics(all_labels, all_preds, class_names: list, output_dir: str):
    import matplotlib.pyplot as plt
    from sklearn.metrics import precision_recall_fscore_support
 
    precision, recall, f1, support = precision_recall_fscore_support(
        all_labels, all_preds, labels=range(len(class_names)), zero_division=0
    )
 
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(class_names))
    width = 0.25
 
    bars1 = ax.bar(x - width, precision, width, label="Precision", color="#E2B08C")
    bars2 = ax.bar(x, recall, width, label="Recall", color="#C97B63")
    bars3 = ax.bar(x + width, f1, width, label="F1-score", color="#8C9EE2")
 
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            h = bar.get_height()
            ax.annotate(f"{h:.2f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)
 
    ax.set_xlabel("Class")
    ax.set_ylabel("Score")
    ax.set_title("Per-class Precision / Recall / F1 (Test Set)")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{c}\n(n={s})" for c, s in zip(class_names, support)])
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
 
    plt.tight_layout()
    path = os.path.join(output_dir, "03_per_class_metrics.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {path}")
 
    return precision, recall, f1, support
 
 
def plot_imbalance_vs_recall(support, recall, class_names: list, output_dir: str):
    import matplotlib.pyplot as plt
    fig, ax1 = plt.subplots(figsize=(10, 6))
 
    ax1.bar(class_names, support, color="#D9CFC1", alpha=0.7, label="Test set support")
    ax1.set_xlabel("Class")
    ax1.set_ylabel("Support (# samples)", color="#8C7B6C")
    ax1.tick_params(axis="y", labelcolor="#8C7B6C")
 
    ax2 = ax1.twinx()
    ax2.plot(class_names, recall, marker="o", color="#C0392B", linewidth=2.5, markersize=8)
    ax2.set_ylabel("Recall", color="#C0392B")
    ax2.tick_params(axis="y", labelcolor="#C0392B")
    ax2.set_ylim(0, 1.0)
    for i, r in enumerate(recall):
        ax2.annotate(f"{r:.2f}", (i, r), textcoords="offset points", xytext=(0, 10),
                     ha="center", color="#C0392B", fontweight="bold")
 
    ax1.set_title("Class Imbalance vs Recall")
    fig.tight_layout()
    path = os.path.join(output_dir, "04_imbalance_vs_recall.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {path}")
 
 
def plot_yolo_style_results(history: dict, freeze_epochs: int, best_epoch: int, output_dir: str):
    """仿 YOLO results.png 風格：一張圖網格顯示所有指標隨 epoch 的折線走勢"""
    import matplotlib.pyplot as plt
    epochs = history["epoch"]
    best_idx = epochs.index(best_epoch)
 
    panels = [
        ("train_loss", "Train Loss", "#E2B08C"),
        ("val_loss", "Val Loss", "#8C9EE2"),
        ("val_precision_macro", "Val Precision (macro)", "#C97B63"),
        ("val_recall_macro", "Val Recall (macro)", "#6BA383"),
        ("val_f1_macro", "Val F1 (macro)", "#C0392B"),
        ("val_accuracy", "Val Accuracy", "#8E6C88"),
    ]
 
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.flatten()
 
    for ax, (key, title, color) in zip(axes, panels):
        values = history[key]
        ax.plot(epochs, values, marker="o", markersize=4, color=color, linewidth=1.8)
        # 用平滑線 (簡單移動平均) 疊加，YOLO 風格會有一條淡色 raw + 一條平滑趨勢線
        if len(values) >= 5:
            window = 3
            smoothed = np.convolve(values, np.ones(window) / window, mode="valid")
            smooth_epochs = epochs[window - 1:]
            ax.plot(smooth_epochs, smoothed, color=color, linewidth=2.5, alpha=0.9, linestyle="--")
        ax.axvline(x=freeze_epochs, color="gray", linestyle=":", alpha=0.5)
        ax.axvline(x=best_epoch, color="green", linestyle=":", alpha=0.7)
        ax.scatter([best_epoch], [values[best_idx]], color="green", s=60, zorder=5)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Epoch", fontsize=9)
        ax.grid(alpha=0.3)
 
    fig.suptitle(
        f"Training Results  (backbone unfreeze @ epoch {freeze_epochs}, best checkpoint @ epoch {best_epoch})",
        fontsize=13, fontweight="bold"
    )
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    path = os.path.join(output_dir, "00_results_grid.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {path}")
 
 
def plot_summary_table(all_labels, all_preds, class_names: list, output_dir: str):
    """產出各類別 + 整體彙總的表格圖與CSV，方便直接複製進報告"""
    import matplotlib.pyplot as plt
    from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                                  precision_score, recall_score, f1_score)
 
    # --- 各類別指標 ---
    precision, recall, f1, support = precision_recall_fscore_support(
        all_labels, all_preds, labels=range(len(class_names)), zero_division=0
    )
 
    rows = []
    for i, name in enumerate(class_names):
        rows.append([name, f"{precision[i]:.3f}", f"{recall[i]:.3f}", f"{f1[i]:.3f}", str(support[i])])
 
    # --- 整體彙總指標 (合一數據) ---
    accuracy = accuracy_score(all_labels, all_preds)
    total_support = len(all_labels)
 
    macro_p = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    macro_r = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
 
    weighted_p = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    weighted_r = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
 
    rows.append(["", "", "", "", ""])  # 空行分隔各類別與整體彙總
    rows.append(["Accuracy", "", "", f"{accuracy:.3f}", str(total_support)])
    rows.append(["Macro Avg", f"{macro_p:.3f}", f"{macro_r:.3f}", f"{macro_f1:.3f}", str(total_support)])
    rows.append(["Weighted Avg", f"{weighted_p:.3f}", f"{weighted_r:.3f}", f"{weighted_f1:.3f}", str(total_support)])
 
    columns = ["Class", "Precision", "Recall", "F1-score", "Support"]
 
    # --- 存成 CSV，方便直接開Excel或複製貼上 ---
    csv_path = os.path.join(output_dir, "05_summary_table.csv")
    with open(csv_path, "w", encoding="utf-8-sig") as f:
        f.write(",".join(columns) + "\n")
        for r in rows:
            f.write(",".join(r) + "\n")
    print(f"[table] 儲存: {csv_path}")
 
    # --- 存成表格圖，方便直接放進簡報/報告 ---
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(rows) + 1.5))
    ax.axis("off")
 
    table = ax.table(cellText=rows, colLabels=columns, loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.6)
 
    # 標題列樣式
    for j in range(len(columns)):
        table[0, j].set_facecolor("#4A4A4A")
        table[0, j].set_text_props(color="white", fontweight="bold")
 
    # 各類別列 (前 len(class_names) 列，index+1因為有標題列)
    for i in range(len(class_names)):
        for j in range(len(columns)):
            table[i + 1, j].set_facecolor("#F5F0EB")
 
    # 整體彙總列 (最後3列，跳過空行) 用不同底色凸顯
    summary_start = len(class_names) + 2  # +1標題列 +1空行
    for i in range(summary_start, summary_start + 3):
        for j in range(len(columns)):
            table[i, j].set_facecolor("#D9E4DD")
            table[i, j].set_text_props(fontweight="bold")
 
    ax.set_title("Classification Summary: Per-class + Overall (Test Set)",
                  fontsize=12, fontweight="bold", pad=15)
 
    plt.tight_layout()
    img_path = os.path.join(output_dir, "05_summary_table.png")
    plt.savefig(img_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {img_path}")
 
 

def generate_all_plots(history: dict, freeze_epochs: int, best_epoch: int,
                        test_metrics: dict, all_labels, all_preds,
                        class_names: list, output_dir: str):
    """訓練/評估完成後一次呼叫，產出全部圖表並存到 output_dir"""
    setup_cjk_font()
    os.makedirs(output_dir, exist_ok=True)
 
    plot_yolo_style_results(history, freeze_epochs, best_epoch, output_dir)
    plot_training_curves(history, freeze_epochs, best_epoch, output_dir)
    plot_confusion_matrix(test_metrics["cm"], class_names, output_dir)
    precision, recall, f1, support = plot_per_class_metrics(all_labels, all_preds, class_names, output_dir)
    plot_imbalance_vs_recall(support, recall, class_names, output_dir)
    plot_summary_table(all_labels, all_preds, class_names, output_dir)
    print(f"\n所有圖表已儲存至: {output_dir}")

def print_overall_summary(backbone_name: str, test_metrics: dict):
    """印出簡潔的整體結果彙總 (純文字版，方便複製貼上到報告/紀錄)"""
    accuracy = test_metrics["val_accuracy"]
    precision = test_metrics["val_precision_macro"]
    recall = test_metrics["val_recall_macro"]
    f1 = test_metrics["val_f1_macro"]

    header = f"[{backbone_name}] 整體結果"
    print("=" * 10 + " " + header)
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print("=" * (len(header) + 11))

print('S')

S


In [21]:
# BLOCK U: 執行入口
# ============================================================

# ------------------------------------------------------------
# 使用方式
# ------------------------------------------------------------
#
# 直接 Run All 即可。
#
#
# 【第一次訓練】
#
# 找不到 checkpoint
#
# ↓
#
# Epoch 1 開始
#
#
# ------------------------------------------------------------
#
# 【同一個 Kaggle Session 重跑】
#
# /kaggle/working/checkpoints/
#
# 已經存在：
#
# *_last.pth
#
# ↓
#
# 程式自動 Resume
#
#
# ------------------------------------------------------------
#
# 【新的 Kaggle Session】
#
# 1.
# 將上一輪：
#
# /kaggle/working/checkpoints/
#
# 保存成 Kaggle Notebook Output。
#
#
# 2.
# 開新的 Notebook Version。
#
#
# 3.
# 將上一輪 Output
# 加成新的 Kaggle Input。
#
#
# 4.
# Run All。
#
#
# 5.
# 程式會自動在：
#
# /kaggle/input/
#
# 裡面遞迴尋找：
#
# EXPERIMENT_NAME_last.pth
#
#
# 6.
# 自動從下一個 Epoch 繼續。
#
#
# ------------------------------------------------------------
#
# 重要：
#
# EPOCHS 是：
#
# 「整個實驗總目標 Epoch」
#
# 不是：
#
# 「每一個 Kaggle Session 都再跑一次」
#
#
# 例如：
#
# EPOCHS = 50
#
# 上次 checkpoint：
#
# Epoch 31
#
# ↓
#
# 下一次會從：
#
# Epoch 32
#
# ↓
#
# 最多跑到：
#
# Epoch 50
#
#
# ------------------------------------------------------------

if __name__ == "__main__":

    main(
        CONFIG
    )

Device: cuda
[resume] 沒找到舊 checkpoint，將從 epoch 1 開始。
[clean_age] 移除 38 筆缺少年齡的紀錄 (16272 -> 16234)
總影像數: 16105, 類別分布:
label
NORM    9012
MI      2509
STTC    2376
CD      1676
HYP      532
Name: count, dtype: int64
train/val/test = 11299/2394/2412

ECG IMAGE AUGMENTATION CONFIG
Enabled            : True
Profile            : liu2026
Rotation           : ±3.0°
Translation        : ±5.0% / ±5.0%
Horizontal Flip    : p=0.0 (ECG 不建議)
Vertical Flip      : p=0.0 (ECG 不建議)

[model] 建立 CBAM-ResNet18 (reduction=16, spatial_kernel=7)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 158MB/s]



開始 / 接續訓練 cbam_resnet18
Start epoch : 1
Target epoch: 50
Backbone 仍在 freeze 階段；將於 epoch 6 開始完整 fine-tune。
  batch 0/354 loss=1.5958
  batch 50/354 loss=1.4601
  batch 100/354 loss=1.5189
  batch 150/354 loss=1.3899
  batch 200/354 loss=1.4819
  batch 250/354 loss=1.4423
  batch 300/354 loss=1.3225
  batch 350/354 loss=1.3307
[cbam_resnet18] Epoch 01/50 | Loss: 1.4445/1.3782 | Acc: 0.5150/0.5067 | Prec: 0.3702 | Rec: 0.3137 | F1: 0.2793 | Time: 815.9s
  → ⭐ Best checkpoint 已寫入硬碟：/kaggle/working/checkpoints/cbam_resnet18_age_sex_liu2026_best.pth
  → ⭐ 新最佳 val_f1_macro: 0.2793
  → 💾 Last checkpoint 已保存：epoch 1
     /kaggle/working/checkpoints/cbam_resnet18_age_sex_liu2026_last.pth
  batch 0/354 loss=1.4011
  batch 50/354 loss=1.1979
  batch 100/354 loss=1.3488
  batch 150/354 loss=1.2947
  batch 200/354 loss=1.4171
  batch 250/354 loss=1.4531
  batch 300/354 loss=1.7446
  batch 350/354 loss=1.5722
[cbam_resnet18] Epoch 02/50 | Loss: 1.3671/1.3552 | Acc: 0.5139/0.4662 | Prec: 0.3777 | Rec

/tmp/ipykernel_23/2808040556.py:73: UserWarning: Glyph 27169 (\N{CJK UNIFIED IDEOGRAPH-6A21}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_23/2808040556.py:73: UserWarning: Glyph 22411 (\N{CJK UNIFIED IDEOGRAPH-578B}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_23/2808040556.py:73: UserWarning: Glyph 37325 (\N{CJK UNIFIED IDEOGRAPH-91CD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_23/2808040556.py:73: UserWarning: Glyph 40670 (\N{CJK UNIFIED IDEOGRAPH-9EDE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_23/2808040556.py:73: UserWarning: Glyph 38364 (\N{CJK UNIFIED IDEOGRAPH-95DC}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_23/2808040556.py:73: UserWarning: Glyph 27880 (\N{CJK UNIFIED IDEOGRAPH-6CE8}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_23/2808040556.py:73: UserWarning: Glyph 21312 (\N{CJK UNIFIED IDEOGRAPH-5340}) missing from

[gradcam] 儲存: /kaggle/working/outputs/gradcam_CD_0.png
[gradcam] 儲存: /kaggle/working/outputs/gradcam_HYP_0.png
[gradcam] 儲存: /kaggle/working/outputs/gradcam_MI_0.png
[gradcam] 儲存: /kaggle/working/outputs/gradcam_NORM_0.png
[gradcam] 儲存: /kaggle/working/outputs/gradcam_STTC_0.png

TRAINING FINISHED
Experiment : cbam_resnet18_age_sex_liu2026
Best epoch : 26
Best metric: 0.6595645356980938
Checkpoint : /kaggle/working/checkpoints
Outputs    : /kaggle/working/outputs
